# Run the full F1 pit-stop pipeline on Colab

Runs every phase in order: download → clean → features → Stage 1 → Stage 2, then (when you switch it on)
the final test on the held-out seasons.

1. **Runtime → Change runtime type → T4 GPU** (Stage 2's neural models train much faster on a GPU).
2. Set `REPO_URL` below to your GitHub repository, or leave it empty and upload `f1-pit-engine.zip` when asked.
3. **Runtime → Run all.**

Downloaded races and results are stored in Google Drive (`MyDrive/f1_project/pipeline`), so a second run
reuses them and only does the work that is missing. The first full download takes a long time.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess, shutil

REPO_URL = ""   # e.g. "https://github.com/<you>/f1-pit-engine.git"; empty = upload f1-pit-engine.zip
QUICK = True    # True = first 4 races per season (a test run of a few minutes); False = full seasons
REPLACE_CODE = False   # True = throw away the code copy in this session and load it again (after a new zip)

os.chdir("/content")
if REPLACE_CODE and os.path.exists("f1-pit-engine"):
    shutil.rmtree("f1-pit-engine")   # the data and results folders are links to Google Drive and are kept
if not os.path.exists("f1-pit-engine"):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, "f1-pit-engine"], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.endswith(".zip"))
        subprocess.run(["unzip", "-oq", zip_name], check=True)
os.chdir("/content/f1-pit-engine")
print("Code is in", os.getcwd())

In [ ]:
!pip install -q -e ".[dl]"
import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
import yaml

# keep data and results in Google Drive so they survive the session
DRIVE_DIR = "/content/drive/MyDrive/f1_project/pipeline"
for folder in ["data", "results"]:
    target = f"{DRIVE_DIR}/{folder}"
    os.makedirs(target, exist_ok=True)
    if os.path.islink(folder):
        continue
    if os.path.exists(folder):
        print(f"'{folder}' already exists in the repo copy; using it instead of Google Drive")
        continue
    os.symlink(target, folder)

config = yaml.safe_load(open("configs/default.yaml"))
config["quick"] = QUICK
with open("configs/colab.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
print("Settings written to configs/colab.yaml (quick =", QUICK, ")")

## Run the pipeline

**Where the races come from.** The download step tries FastF1 first. The Formula 1 timing service refuses
many requests from cloud computers such as Colab, so when FastF1 returns nothing the race is read from the
TracingInsights archive on GitHub instead: exports of the same FastF1 lap tables, which Colab can reach. The
step prints how many races came from each source; say so in your report. Lines starting with `MISS` are
races neither source has yet (usually the most recent ones); the pipeline carries on without them.

Each step runs only if the one before it finished. If a step stops, read its last lines, fix the cause and
run the cell again: races already saved are reused.

In [ ]:
import subprocess, sys

def run_step(module, *extra):
    """Run one pipeline step, show its output as it arrives, and stop the cell if the step fails."""
    print(f"\n===== {module} =====", flush=True)
    command = [sys.executable, "-u", "-m", f"f1pit.{module}", "--config", "configs/colab.yaml", *extra]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"The '{module}' step stopped with an error (see its last lines above). "
                           "The later steps were not run. Fix the cause, then run this cell again.")

for step in ["ingest", "clean", "features", "stage1"]:
    run_step(step)

In [ ]:
# Stage 2 trains every experiment with every seed. To run only some, use for example:
#   run_step("stage2", "--experiments", "rule_based", "xgboost", "lstm_tyre_aware", "lstm_no_stage1")
run_step("stage2")

## Results

In [ ]:
import json
import pandas as pd

results = json.load(open("results/stage2_metrics.json"))
rows = []
for name, e in results["experiments"].items():
    m = e["seed_ensemble"]
    rows.append({"experiment": name, "PR-AUC": m["pr_auc"], "CI low": m["pr_auc_ci"]["ci_low"],
                 "CI high": m["pr_auc_ci"]["ci_high"], "F1": m["f1"], "recall": m["recall"],
                 "pit within ±2 laps": m["pit_within_window"], "no-skill PR-AUC": m["no_skill_pr_auc"]})
display(pd.DataFrame(rows).set_index("experiment").round(3))

tests = []
for label, c in results["comparisons"].items():
    d = c["pr_auc_difference"]
    tests.append({"comparison": label, "A": c["model_a"], "B": c["model_b"],
                  "PR-AUC A - B": d["estimate"], "CI low": d["ci_low"], "CI high": d["ci_high"],
                  "bootstrap p": d["p_not_better"], "McNemar p": c["mcnemar"]["p_value"]})
display(pd.DataFrame(tests).set_index("comparison").round(3))

**Reading the hypothesis rows:** a comparison supports "A is better than B" when the whole 95% interval of
`PR-AUC A - B` is above zero (bootstrap p below 0.05). An interval that crosses zero means the data cannot
tell the two models apart. That is a valid finding, and it should be reported and discussed, not hidden.

## Final test on the held-out seasons (run once)

Everything above used the training and validation seasons only. This section scores the frozen models
on the 2025 test season and the 2026 extra-test season. Nothing is fitted or tuned here.

**Leave `RUN_FINAL_TEST = False` until your models and settings are final**, then set it to `True` and
run the cells below. Every evaluation is recorded in `results/final_evaluation_log.json`; running again
on the same models only shows the saved results. If you retrain and evaluate again, say so in the report.

In [ ]:
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    print("Final test not run. Set RUN_FINAL_TEST = True when the models are final.")
elif QUICK:
    print("Final test not run: QUICK = True trains on only a few races. Set QUICK = False, run the whole "
          "notebook again, then run this cell.")
else:
    run_step("evaluate")

In [ ]:
import os
from IPython.display import Image

if os.path.exists("results/final_model_table.csv"):
    table = pd.read_csv("results/final_model_table.csv")
    cols = ["split", "subset", "experiment", "pr_auc", "pr_auc_ci_low", "pr_auc_ci_high", "f1", "recall",
            "pit_within_window", "brier_after_platt", "ece_after_platt", "n_races"]
    display(table[cols].round(3))
    display(pd.read_csv("results/final_hypothesis_table.csv").round(3))
    print("Evaluations of the test seasons so far:", len(json.load(open("results/final_evaluation_log.json"))))
    for name in sorted(os.listdir("results/figures")):
        print(name)
        display(Image(f"results/figures/{name}", width=750))
else:
    print("No final results yet.")

**Reading the final results:** the headline rows are `subset = excluding_rule_forced` (races where stops
were forced by a special rule are left out, because no model was trained on such races); `all_races` is
shown next to them as a check. The final test needs a full run (`QUICK = False`): a few races per season
cannot support a conclusion.

## Report pack

Writes the tables and data facts for your written report into `results/report/` and downloads them, with the
charts, as `report_pack.zip`. Copy numbers into the report from these files only.

In [ ]:
if os.path.exists("results/final_metrics.json"):
    !python -m f1pit.report --config configs/colab.yaml
    pack = "/content/report_pack"
    shutil.rmtree(pack, ignore_errors=True)
    shutil.copytree("results/report", pack + "/report")
    if os.path.isdir("results/figures"):
        shutil.copytree("results/figures", pack + "/figures")
    for name in ["final_model_table.csv", "final_hypothesis_table.csv", "final_evaluation_log.json"]:
        if os.path.exists(f"results/{name}"):
            shutil.copy(f"results/{name}", pack)
    shutil.make_archive(pack, "zip", root_dir=pack)
    from google.colab import files
    files.download(pack + ".zip")
else:
    print("Run the final test first.")

## Dashboard data

After the final test, this builds the small data bundle the dashboard shows and downloads it as
`dashboard_data.zip`. Unzip it into the `app/` folder of your repository (so the files are in `app/data/`),
commit it, and the deployed dashboard shows your results instead of the practice data.

In [ ]:
if os.path.exists("results/final_metrics.json"):
    !python -m f1pit.dashboard --config configs/colab.yaml --out app/data
    shutil.make_archive("/content/dashboard_data", "zip", root_dir="app", base_dir="data")
    from google.colab import files
    files.download("/content/dashboard_data.zip")
else:
    print("Run the final test first.")